# Analyze Study (Binary Classification)

This notebook provides a comprehensive post-hoc analysis of a completed Octopus binary classification study. It covers:

1. **Study Details** -- load, validate, and summarize the study configuration and workflow structure.
2. **Development Performance** -- metric scores across outer splits for all prediction tasks and result types. Use this to guide model selection and hyperparameter decisions.
3. **Selected Features** -- number of selected features per task and outer split, plus feature selection frequency.
4. **Test Set Evaluation** -- for a selected task: test-set metrics (per-split, mean, and merged), AUCROC curves, confusion matrices, and feature importances (permutation and SHAP).

**Prerequisite:** Run the multi-step classification workflow first to generate study data:

```bash
python examples/wf_multi_step_classification.py
```

## Imports

In [ ]:
from octopus.poststudy import OctoTestEvaluator, load_study_information
from octopus.poststudy.analysis.notebook import (
    display_confusionmatrix,
    display_feature_groups_table,
    display_performance_tables,
    display_study_overview,
)
from octopus.poststudy.analysis.plots import (
    aucroc_per_split_plot,
    aucroc_plot,
    dev_performance_plot,
    feature_count_plot,
    feature_frequency_plot,
    fi_plot,
    performance_plot,
)
from octopus.poststudy.analysis.tables import (
    aucroc_data,
    fi_ensemble_table,
    get_performance,
    get_selected_features,
    workflow_graph,
)
from octopus.poststudy.study_io import find_latest_study

## Input

For the analysis, a path to a saved study directory needs to be provided. Octopus saves studies in directories named `<prefix>-YYYYMMDD_HHMMSS`. The helper `find_latest_study` scans the studies root folder and returns the most recent directory matching the given name prefix. Overwrite study_directory if you want to use a specific study path.

In [ ]:
study_name_prefix = "wf_multi_step_classification"
study_directory = find_latest_study("../studies", study_name_prefix)
print(f"Using study: {study_directory}")

## Study Details

`load_study_information` validates the study directory (checks outersplit and task directories) and returns a `StudyInfo` object used by all downstream analysis functions. `display_study_overview` prints a concise summary. `workflow_graph` renders the task dependency tree.

In [ ]:
study_info = load_study_information(study_directory)
display_study_overview(study_info)

In [ ]:
print(workflow_graph(study_info))

## Development Performance

**What it shows:** Metric scores per outer split for all prediction tasks in the workflow. Feature selection tasks (mrmr, roc, boruta) are skipped automatically. If a task has multiple result types (e.g. `best` and `ensemble_selection`), each is shown separately. By default the `dev` partition is shown -- use this partition for model selection and hyperparameter decisions.

**What to look for:**
- Consistent bar heights across splits → stable model performance
- Large variation between splits → model sensitive to the train/test partition (common with small datasets or class imbalance)
- Comparing bars across tasks → reveals whether feature selection improves or degrades performance
- Use the metric dropdown to switch between available metrics

In [ ]:
perf = get_performance(study_info)

In [ ]:
fig = dev_performance_plot(perf)
fig.show()

In [ ]:
display_performance_tables(perf)

## Selected Features

**What it shows:** Number of selected features per outer split and task, plus how frequently each feature was selected across splits. Covers all tasks and result types discovered on disk.

**What to look for:**
- **Feature count plot:** Consistent count across splits → stable feature selection. A drop between workflow tasks confirms that feature selection steps are reducing dimensionality as intended.
- **Feature frequency plot:** Features selected in all splits are robust and likely genuinely informative. Features selected in only one or two splits may be noise or split-specific artefacts.

In [ ]:
feature_table, frequency_table = get_selected_features(study_info)

In [ ]:
fig = feature_count_plot(feature_table)
fig.show()

In [ ]:
fig = feature_frequency_plot(frequency_table)
fig.show()

## Test Set Evaluation

The sections above use the **dev** partition -- these results should guide model selection, hyperparameter tuning, and feature selection decisions. Looking at test scores during these steps introduces data leakage and inflates reported performance.

The sections below evaluate a **single selected task** on the held-out **test** partition. Only look at test results **after all modelling decisions have been made**. If test results lead you to change the model or features, the test set is no longer independent and the reported scores lose their validity.

`OctoTestEvaluator` loads the fitted models and the stored train/test splits for the selected task. All subsequent calls (performance, AUCROC, confusion matrix, feature importance) use this evaluator.

### Test Performance

**What it shows:** Test-set metrics per outer split (each model scored only on its own held-out test fold), plus a ``Mean`` row (average of per-split scores) and a ``Merged`` row (all test predictions pooled across splits and scored as one set). All metrics compatible with the ML type are computed automatically.

**What to look for:**
- The **Merged** row is the standard "pooled out-of-fold" evaluation — it treats all held-out predictions as a single test set. This is typically the most robust summary because it uses all samples without averaging away variance
- The **Mean** row averages per-split scores — it can differ from Merged, especially for nonlinear metrics (e.g. AUCROC) or when splits have unequal sizes
- Use the metric dropdown to compare different metrics across splits
- Large spread across splits → model performance depends on the data partition

In [ ]:
test_evaluator = OctoTestEvaluator(
    study_info=study_info, task_id=0, result_type="best"
)
test_perf = test_evaluator.performance()
performance_plot(test_perf).show()

In [ ]:
test_perf.round(3)

### AUCROC Curves

**What it shows:** ROC (Receiver Operating Characteristic) curves on the held-out test data. The ROC curve plots the True Positive Rate (sensitivity) against the False Positive Rate (1 - specificity) at different classification thresholds. The Area Under the Curve (AUC) summarises the model's ability to distinguish between classes.

The first plot shows both the merged curve (all predictions pooled) and the averaged curve (mean +/- 1 std). The second plot shows individual per-split curves.

**What to look for:**
- Curve hugging the top-left corner → strong discrimination (AUC close to 1.0)
- Dashed diagonal = random guessing (AUC = 0.5) — your model should be well above this line
- AUC above 0.8 is generally good, above 0.9 excellent (domain-dependent)
- Narrow std band in the averaged plot → consistent performance across splits; wide band → performance varies by split
- Per-split curves that differ substantially → investigate whether specific splits have unusual data distributions

In [ ]:
roc = aucroc_data(test_evaluator)
aucroc_plot(roc).show()

In [ ]:
aucroc_per_split_plot(roc).show()

### Confusion Matrix

**What it shows:** Per-split confusion matrices (absolute counts and relative percentages) with a metrics panel. Rows represent true class labels, columns represent predicted labels. Each outer split is shown separately.

**What to look for:**
- Diagonal cells = correct predictions. High values on the diagonal → model classifies well
- Off-diagonal cells = misclassifications. A high value in row "A", column "B" → model frequently predicts B when the true class is A
- Relative view: each row sums to 100% — the diagonal value is the recall for that class
- Compare splits to check whether misclassification patterns are consistent or split-specific
- The metrics panel shows per-split performance scores rounded to 3 digits

In [ ]:
display_confusionmatrix(test_evaluator, threshold=0.5)

### Feature Importance — Permutation

**What it shows:** Permutation importance on the held-out test data. Each feature is randomly shuffled and the resulting performance drop is measured. A large drop means the feature is important. `group_permutation` also computes importance at the feature-group level, revealing whether correlated features contribute collectively.

Each outer-split model is evaluated independently on its own test fold, then results are aggregated into an ensemble summary.

**What to look for:**
- Features sorted by importance — top features contribute most to predictions
- `n_repeats=3` is used here for speed; use 10+ for real analyses to get stable estimates and reliable p-values

In [ ]:
fi_table_perm = test_evaluator.calculate_fi(
    fi_type="group_permutation", n_repeats=3
)
fi_plot(fi_table_perm).show()

In [ ]:
_ = display_feature_groups_table(test_evaluator)

In [ ]:
fi_ensemble_table(fi_table_perm).head(10)

### Feature Importance — SHAP

**What it shows:** SHAP (SHapley Additive exPlanations) values computed on the held-out test data. SHAP attributes each prediction to individual features using game-theoretic Shapley values. More fine-grained than permutation importance but slower.

**What to look for:**
- Features sorted by mean |SHAP value| — top features have the largest average impact on predictions
- Available `shap_type` options: `kernel` (default, model-agnostic), `permutation`, `exact` (slowest, most accurate)
- Compare with permutation results — features that rank highly in both methods are reliably important

In [ ]:
fi_table_shap = test_evaluator.calculate_fi(
    fi_type="shap", shap_type="kernel"
)
fi_plot(fi_table_shap).show()

In [ ]:
fi_ensemble_table(fi_table_shap).head(10)